# Chapter 18 &mdash; Using Fixpoint Combinators: Factorial and Fibonacci in Python

**Concept 9 of the Chapter 18 decomposition:** *Using Fixpoint Combinators: Factorial and Fibonacci in Python*

Write the recursion as a non-recursive $G$, then apply the combinator.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Factorial-And-Fibonacci/Concept-Factorial-And-Fibonacci.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The recipe, in three steps:

1. write the recursive function normally;
2. **abstract over the recursive call** &mdash; replace the self-reference by a
   parameter $f$, giving a non-recursive $G$;
3. apply $Y_e$.

```python
G    = lambda f: lambda n: 1 if n == 0 else n * f(n-1)
fact = Ye(G)
```

The result is a genuinely anonymous recursive function. Practically you would just
write `def`, but the exercise shows that **recursion is not a primitive** &mdash; it is
derivable from abstraction and application alone.

Memoisation drops in at the $G$ level, which is a small but real payoff: you wrap the
`f` you hand to $G$ and get a memoised recursion without touching the definition.

## 2. Definitions

### The combinator and several G's

In [ ]:
# --- fixpoint combinators ------------------------------------------------
# Y diverges under Python's EAGER evaluation, because (x x) is evaluated
# before it is needed.  Y_e ("eager Y", also called Z) wraps the
# self-application in a lambda, delaying it until it is applied.
Y  = lambda f: (lambda x: f(x(x)))(lambda x: f(x(x)))          # loops in Python
Ye = lambda f: (lambda x: f(lambda v: x(x)(v)))(lambda x: f(lambda v: x(x)(v)))


G_fact  = lambda f: lambda n: 1 if n == 0 else n * f(n - 1)
G_fib   = lambda f: lambda n: n if n < 2 else f(n - 1) + f(n - 2)
G_gcd   = lambda f: lambda a: lambda b: a if b == 0 else f(b)(a % b)
G_rev   = lambda f: lambda s: '' if not s else s[-1] + f(s[:-1])
G_ack   = lambda f: lambda m: lambda n: (n + 1 if m == 0 else
                                         f(m - 1)(1) if n == 0 else
                                         f(m - 1)(f(m)(n - 1)))

### Memoisation, inserted at the G level

In [ ]:
def memoize_fix(G):
    cache = {}
    def rec(n):
        if n not in cache:
            cache[n] = G(rec)(n)
        return cache[n]
    return rec, cache

## 3. Tests

Factorial and Fibonacci, with no names in the definition.

In [ ]:
fact, fib = Ye(G_fact), Ye(G_fib)
print("  fact :", [fact(n) for n in range(9)])
print("  fib  :", [fib(n) for n in range(11)])
assert fact(8) == 40320
assert [fib(n) for n in range(11)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]

Two-argument recursion, by currying.

In [ ]:
gcd = Ye(G_gcd)
for a, b in [(12, 18), (100, 75), (17, 5)]:
    import math
    print("  gcd(%d, %d) = %d  (math.gcd says %d)" % (a, b, gcd(a)(b), math.gcd(a, b)))
    assert gcd(a)(b) == math.gcd(a, b)

Recursion over strings, not just numbers.

In [ ]:
rev = Ye(G_rev)
for s in ['', 'a', 'abc', 'lambda']:
    print("  rev(%-8r) = %r" % (s, rev(s)))
    assert rev(s) == s[::-1]

Ackermann, to show the combinator does not care how wild the recursion is.

In [ ]:
ack = Ye(G_ack)
for m in range(3):
    print("  A(%d, n) for n=0..4 :" % m, [ack(m)(n) for n in range(5)])
assert ack(2)(3) == 9
assert ack(3)(2) == 29

**Memoisation at the $G$ level** &mdash; the definition is untouched.

In [ ]:
import time
slow = Ye(G_fib)
fastr, cache = memoize_fix(G_fib)
t0 = time.time(); a = slow(24); t1 = time.time()
t2 = time.time(); b = fastr(24); t3 = time.time()
print("  fib(24) = %d" % a)
print("  naive     : %.4fs" % (t1 - t0))
print("  memoized  : %.6fs  (cache holds %d entries)" % (t3 - t2, len(cache)))
assert a == b == 46368
assert (t3 - t2) < (t1 - t0)
print("\nG_fib was not modified.  Only the f handed to it changed.")

The recipe, one more time.

In [ ]:
print("1. write it recursively")
print("2. replace the self-reference by a parameter  -> G")
print("3. apply Ye")
print()
print("Recursion is not a language primitive.  It is abstraction plus")
print("application plus one very strange combinator.")

## 4. Exercises


1. Write $G$ for mutual recursion (`even`/`odd`). What does the fixpoint look like?
2. Add memoisation to `ack`. How much does it help?
3. Why does `memoize_fix` work without knowing anything about $G$?

In [ ]:
# Your work for the exercises above.